# Visualization of Tool Results on Reuters-50-50 Dataset

In [33]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import ast
import json

Load Data

In [34]:
df = pd.read_parquet("data/reuters_processed.parquet")

Charts

In [35]:
def robust_parse(val):
    """robust parse algorithm json to dict"""
    if not val:
        return {}
    if isinstance(val, dict):
        return val
    if isinstance(val, str):
        try:
            parsed = json.loads(val)
            if isinstance(parsed, str):
                return ast.literal_eval(parsed)
            return parsed
        except json.JSONDecodeError:
            try:
                return ast.literal_eval(val)
            except:
                return {}
    return {}

def extract_error_rate(val):
    data = robust_parse(val)
    return data.get("errors_per_100_words", data.get("typos_per_100_words", 0))

def extract_comma_freq(val):
    data = robust_parse(val)
    return data.get(",", 0)

def extract_pos_dominance(val):
    data = robust_parse(val)
    return sum(data.values()) if isinstance(data, dict) else 0

print("Extrahiere JSON-Metriken...")
df["error_rate"] = df["typical_error_patterns_json"].apply(extract_error_rate)
df["comma_freq"] = df["punctuation_frequency_json"].apply(extract_comma_freq)
df["pos_dominance"] = df["pos_ngrams_json"].apply(extract_pos_dominance)

Extrahiere JSON-Metriken...


In [36]:
features = [
    ("lexical_diversity", "Tool 1: Lexikalische Vielfalt (Ratio)"),
    ("average_sentence_length", "Tool 2: Durchschnittliche Satzlänge (Wörter)"),
    ("comma_freq", "Tool 3: Relative Komma-Häufigkeit"),
    ("function_words_frequency", "Tool 4: Anteil Funktionswörter (Stopwords)"),
    ("pos_dominance", "Tool 5: Top-5 POS N-Gram Häufigkeit"),
    ("sentence_dependency_depth", "Tool 6: Syntaktische Verschachtelungstiefe"),
    ("error_rate", "Tool 7: Fehler & Typos pro 100 Wörter")
]

fig = make_subplots(
    rows=7, cols=1, 
    subplot_titles=[f[1] for f in features],
    vertical_spacing=0.05
)

for i, (col, title) in enumerate(features):
    
    # calculate ascending order
    median_order = df.groupby("author")[col].median().sort_values().index.tolist()
    
    fig.add_trace(
        go.Box(
            x=df["author"], 
            y=df[col], 
            name=col,
            showlegend=False,
            marker_color="#3498db",
            hoverinfo="y+x"
        ),
        row=i+1, col=1
    )
    fig.update_xaxes(
        categoryorder="array", 
        categoryarray=median_order, 
        row=i+1, col=1,
        tickangle=90
    )

fig.update_layout(
    height=4500,
    width=1400,
    title_text="Reuters 50-50: Interaktive Tool-Analyse",
    title_font_size=24,
    plot_bgcolor="white",
    paper_bgcolor="white"
)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

fig.show()